# 50keV Cascades in Fe

This is a benchmark example showing the following techniques:
- running in cascade only mode
- disabling electronic energy loss
- enabling intra-cascade recombination
- using UserTally to obtain information

50 keV Fe cascades are produced isotropically at the center of a 120 nm side Fe cube.

Defect production predicted by damage models:
- NRT: 500 FPs/cascade
- ARC-DPA: 153 FPs

The values are to be compared with total vacancy production obtained by OpenTRIM, with and without intra-cascade recombination.

A UserTally is used to get the radial defect distribution.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import opentrim
print('opentrim', opentrim.__version__)

## 1. Configure

In [ ]:
config = opentrim.Config()

config.Simulation.simulation_type = opentrim.SimulationType.CascadesOnly
config.Simulation.electronic_stopping = opentrim.Stopping.Off

config.Transport.flight_path_type = opentrim.FlightPath.Constant

config.IonBeam.ion = opentrim.Element('Fe')
config.IonBeam.energy_distribution.center = 50000.0  # eV
config.IonBeam.spatial_distribution.geometry = opentrim.Geometry.Volume
config.IonBeam.spatial_distribution.center = [60.0, 60.0, 60.0]
config.IonBeam.angular_distribution.type = opentrim.Distribution.Uniform
config.IonBeam.angular_distribution.fwhm = 4*np.pi # Isotropic 4π

config.Target.size = [120.0, 120.0, 120.0]
config.Target.periodic_bc = [0,0,0]

# material
mat = opentrim.Material(); mat.id = 'Fe'; mat.density = 7.874
atom = opentrim.Atom(); atom.element = opentrim.Element('Fe')
atom.X = 1.0; atom.Ed = 40.0; atom.El = 3.0; atom.Er = 40.0; atom.Rc = 0.9
mat.composition.append(atom)
config.Target.materials.append(mat)
#region
region = opentrim.Region(); region.id = 'cube'; region.material_id = 'Fe'
region.size = [120.0, 120.0, 120.0]
config.Target.regions.append(region)

# UserTally
ut = opentrim.UserTally()
ut.id = "Nfp"
ut.description = "# of Frenkel pairs = # of Vacancies"
ut.event = opentrim.Event.Vacancy
ut.coordinate_system.origin = [60.0, 60.0, 60.0]
r_edges = np.arange(0, 62, 2) # r = sqrt(x^2+y^2+z^2) bin edges, nm
r_centers = 0.5 * (r_edges[:-1] + r_edges[1:]) # bin centers
ds = 4*np.pi*(r_edges[1:]**3 - r_edges[:-1]**3)/3  # bin spherical volumes, nm^3
ut.bins.r = r_edges.tolist()
ut.bins.atom_id = [1.0, 2.0]
config.UserTally.append(ut)

# Run
config.Run.max_no_ions = 10000

config.validate()
config

## 1st Run - no recombination

In [ ]:
sim = opentrim.Driver(config)

sim.run()                       # returns immediately
sim.wait()                      # block until done
print('done, total ions:', sim.ion_count())

# get results
info = opentrim.Info(sim)
v, dv = info['user_tally']['Nfp']['data']
nfp0 = v[0:,0]/ds # normalize to defects per unit volume, nm^-3 
Nfp0 = info['tally']['totals']['data'][0][1][1] # total # of vacancies
print('Nfp = ', Nfp0)

## 2nd Run - with intra-cascade recombination

In [ ]:
config.Simulation.intra_cascade_recombination = True

sim = opentrim.Driver(config)

sim.run()                       # returns immediately
sim.wait()                      # block until done
print('done, total ions:', sim.ion_count())

# get results
info = opentrim.Info(sim)
v, dv = info['user_tally']['Nfp']['data']
nfp1 = v[0:,0]/ds # normalize to defects per unit volume, nm^-3 
Nfp1 = info['tally']['totals']['data'][0][1][1] # total # of vacancies
print('Nfp = ', Nfp1)

## Plot the defect density

In [ ]:
plt.figure(figsize=(8, 4))
plt.semilogy(r_centers, nfp0, label='Nfp initial',marker='o', markersize=4, linewidth=1)
plt.semilogy(r_centers, nfp1, label='Nfp after recombination',marker='o', markersize=4, linewidth=1)
plt.xlabel('r (nm)')
plt.ylabel('Nfp (nm^-3)')
plt.title(f'50 keV cascades in Fe (N={sim.ion_count():,} ions)')
plt.legend()
plt.tight_layout()
plt.show()